## 1. Environment summary

Research-only post-hoc adaptation. Protocol `07c93b4657e84a4ddfbdc2df1af0f467f80959e0534a67840b4bf2b2b04a2c2c`. Free GPU only; the 801-row holdout stays closed until TRAIN-only selection and refits finish.


In [ ]:
from pathlib import Path
import json, os, platform, shutil, sqlite3, subprocess, sys
ROOT = Path('/content/EvoVariant')
DRIVE = Path('/content/drive/MyDrive/EvoVariantTR_original')
BRANCH = 'research/posthoc-foundation-adaptation'
try:
    import torch
    CUDA_READY = bool(torch.cuda.is_available())
    GPU = torch.cuda.get_device_name(0) if CUDA_READY else None
except ImportError:
    CUDA_READY, GPU = False, None
print({'python': sys.version.split()[0], 'platform': platform.platform(), 'cuda': CUDA_READY, 'gpu': GPU})

## 2. Drive mount

Mount the new account's Drive and use the shortcut to the original study folder. The separate EvoVariantTR folder owned by the new account is empty and must not be used.


In [ ]:
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert DRIVE.is_symlink() and '1RMhA2eEUsvgryqz8YnRryuniroDiTA89' in str(DRIVE.resolve()), 'Original Drive shortcut is missing or points elsewhere'
assert all((DRIVE / name).is_dir() for name in ('reference','datasets','checkpoints','model_cache','runs','hpo','logs','state','exports'))
assert (DRIVE / 'hpo/caduceus_hpo.sqlite3').is_file(), 'Original HPO study database is missing'
print('Drive mounted:', DRIVE)

## 3. Repository and branch verification

Preserve any dirty checkout. A clean checkout is fast-forwarded to the adaptation branch; baseline `main` is never modified.


In [ ]:
if not ROOT.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/UtkarsHMer05/EvoVariant-TR-.git',str(ROOT)],check=True)
else:
    dirty = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain'],text=True).strip()
    if dirty:
        raise RuntimeError('Colab checkout has uncommitted changes; preserve them before updating')
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',BRANCH],check=True)
branch = subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'],text=True).strip()
assert branch == BRANCH, branch
head = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
os.environ['PYTHONPATH'] = str(ROOT / 'src')
print('branch:', branch, 'HEAD:', head)

## 4. Persisted state verification

Copy the Drive SQLite file to local disk before checking it. The original database remains untouched.


In [ ]:
STATE = DRIVE / 'state/adaptation_state.json'
if STATE.exists():
    state = json.loads(STATE.read_text())
    print('last persisted stage:', state.get('stage'))
else:
    print('No persisted state file')
db = DRIVE / 'hpo/caduceus_hpo.sqlite3'
snapshot = db.with_name(db.name + '.snapshot')
sources = [path for path in (snapshot, db) if path.exists()]
if sources:
    local = Path('/content/evovariant_runtime/hpo/preflight.sqlite3')
    local.parent.mkdir(parents=True, exist_ok=True)
    for source in sources:
        shutil.copyfile(source, local)
        try:
            with sqlite3.connect(f'{local.as_uri()}?mode=ro', uri=True) as conn:
                integrity = conn.execute('PRAGMA integrity_check').fetchone()[0]
                if integrity != 'ok':
                    raise RuntimeError(integrity)
                trials = conn.execute('SELECT number,state FROM trials ORDER BY number').fetchall()
            print('study DB:', source.name, 'integrity:', integrity, 'trials:', trials)
            break
        except (sqlite3.DatabaseError, RuntimeError) as error:
            print('DB copy needs forensic recovery:', source.name, type(error).__name__)
    else:
        raise RuntimeError('No valid HPO database; preserve artifacts and reconstruct before training')
else:
    print('No study DB yet; the runner will check existing checkpoints before creating one')

## 5. Isolated runtime activation

Reuses verified Python 3.11/CUDA dependencies. CPU sessions can inspect evidence without installing or training.


In [ ]:
PY = None
if CUDA_READY:
    subprocess.run([sys.executable,str(ROOT/'scripts/adaptation/bootstrap_environment.py'),
                    '--root',str(ROOT)],check=True)
    PY = str(Path('/content/caduceus-env/bin/python'))
    print('isolated Python:', PY)
else:
    print('WAITING_FOR_FREE_GPU: environment installation and training skipped')

## 6. Reference and cache verification

Reuse the Drive archive and index; verify the local FASTA hash and all 4,000 formal REF alleles before model work.


In [ ]:
REFERENCE = Path('/content/Homo_sapiens_assembly38.fasta')
if PY:
    env = {**os.environ, 'PYTHONPATH':str(ROOT/'src')}
    subprocess.run([PY,str(ROOT/'scripts/adaptation/prepare_reference.py'),'--root',str(ROOT),
                    '--drive-root',str(DRIVE),'--output',str(REFERENCE)],env=env,check=True)
    data_ready = DRIVE/'state/data_ready.json'
    verified = json.loads(data_ready.read_text()) if data_ready.exists() else {}
    ref_check = verified.get('reference',{})
    if ref_check.get('sha256') != '5be01555d98347fdb3714dc84c6f77c9d8bc774adcf32c6f7a8fa06f5baf5e51' or ref_check.get('checked_formal_ref_alleles') != 4000:
        subprocess.run([PY,str(ROOT/'scripts/adaptation/verify_data.py'),'--root',str(ROOT),
                        '--reference',str(REFERENCE),'--output',str(data_ready)],env=env,check=True)
else:
    print('Reference preparation waits for an eligible free GPU session')
print('model cache directory:', DRIVE/'model_cache')

## 7. Resume status

The first incomplete HPO fold is determined from its verified history/checkpoint, never by rerunning trial 0 from epoch 1.


In [ ]:
FROZEN = DRIVE/'checkpoints/caduceus_frozen_head/run.json'
LOCK = DRIVE/'hpo/selection_closed.json'
print('frozen report:', json.loads(FROZEN.read_text()).get('status') if FROZEN.exists() else 'MISSING')
print('selection:', 'CLOSED' if LOCK.exists() else 'OPEN')
for history in sorted((DRIVE/'checkpoints/caduceus_hpo').glob('*/fold_*/history.json')):
    epochs = json.loads(history.read_text())
    print(history.relative_to(DRIVE), 'saved epochs:', len(epochs))
print('holdout gate:', 'CLOSED' if not LOCK.exists() else 'requires final TRAIN refits and OOF calibration')

## 8. Autonomous runner launch

One guarded source runner advances Caduceus HPO, TRAIN OOF analysis, fixed-seed refits, then the one-shot 801 evaluation only after the lock closes.


In [ ]:
PID_FILE = DRIVE/'state/autonomous_runner.pid'
LOG = DRIVE/'logs/autonomous_runner.log'
def live_worker(pid):
    result = subprocess.run(['ps','-p',str(pid),'-o','args='],capture_output=True,text=True)
    return result.returncode == 0 and 'run_autonomous.py' in result.stdout
if not PY:
    print('WAITING_FOR_FREE_GPU; no worker launched')
elif PID_FILE.exists() and live_worker(PID_FILE.read_text().strip()):
    print('RUNNING:', PID_FILE.read_text().strip())
else:
    env = {**os.environ,'PYTHONPATH':str(ROOT/'src'),'PYTHONUNBUFFERED':'1'}
    with LOG.open('a',encoding='utf-8') as log:
        worker = subprocess.Popen([PY,str(ROOT/'scripts/adaptation/run_autonomous.py'),
                                   '--root',str(ROOT),'--drive-root',str(DRIVE),
                                   '--reference',str(REFERENCE)],cwd=ROOT,env=env,
                                  stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
    PID_FILE.write_text(str(worker.pid))
    print('RUNNING:',worker.pid,'log:',LOG)

## 9. Non-throwing monitor

A finished process is reported as a state, not an exception. Re-run this cell to refresh progress.


In [ ]:
STATUS = DRIVE/'state/autonomous_runner_status.json'
pid = PID_FILE.read_text().strip() if PID_FILE.exists() else None
if pid and live_worker(pid):
    process_state = 'RUNNING'
elif STATUS.exists():
    saved = json.loads(STATUS.read_text()).get('status')
    process_state = ('EXITED_FAILURE' if saved == 'EXITED_FAILURE' else
                     'UNKNOWN' if saved == 'RUNNING' else 'EXITED_SUCCESS')
else:
    process_state = 'UNKNOWN'
print('process:',process_state)
print('runner:',json.loads(STATUS.read_text()) if STATUS.exists() else 'PENDING')
print('log tail:',LOG.read_text(errors='replace')[-3000:] if LOG.exists() else 'PENDING')

## 10. Final summary

Report only persisted evidence. The historical 946-row temporal result is never used for adaptation selection.


In [ ]:
current = json.loads(STATE.read_text()) if STATE.exists() else {}
hpo_report = DRIVE/'hpo/caduceus_hpo.json'
print('state:',current.get('stage','PENDING'))
print('HPO:',json.loads(hpo_report.read_text()).get('status') if hpo_report.exists() else 'PENDING')
print('selection:', 'CLOSED' if LOCK.exists() else 'OPEN')
for seed in (42,1337,2026):
    suffix = '' if seed == 42 else f'_seed{seed}'
    report = DRIVE/f'exports/caduceus_validation{suffix}.json'
    print('801 one-shot seed',seed,':',json.loads(report.read_text()).get('status') if report.exists() else 'PENDING')
print('remaining study stages: NT, robustness, figures, and final report require measured artifacts')